# Classical NLP Methods — AI Lab Instructor Notebook

*CV & NLP Basics · Hard*



9 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np
import re
from collections import Counter, defaultdict

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Classical NLP Methods — Student Lab

You will implement classical NLP features and baselines from scratch:
- Bag-of-Words (counts)
- TF-IDF (Term Frequency-Inverse Document Frequency)
- cosine similarity retrieval
- Naive Bayes + Logistic Regression
- n-grams

Constraints: offline, deterministic, no hardcoded outputs.

## Dataset (offline)
Small labeled corpus: `y=1` positive, `y=0` negative.

FAANG gotcha: even with toy data, enforce a clean train/val split and never fit vocab/IDF on val.

In [ ]:
data = [
    ('I loved the movie it was fantastic and inspiring', 1),
    ('Amazing acting and great story', 1),
    ('This was an excellent film with wonderful music', 1),
    ('I enjoyed it a lot truly great', 1),
    ('The plot was engaging and the visuals were stunning', 1),
    ('Brilliant and uplifting experience', 1),
    ('Terrible movie boring and too long', 0),
    ('I hated it the acting was awful', 0),
    ('Worst film ever waste of time', 0),
    ('The story made no sense and was dull', 0),
    ('Painfully slow and disappointing', 0),
    ('Bad script poor direction', 0),
    # neutral-ish / tricky
    ('Not bad but not great either', 0),
    ('It was okay some parts were good', 1),
    ('I expected more it felt average', 0),
]
texts = [t for t, y in data]
y = np.array([y for t, y in data], dtype=np.int64)
print('n', len(texts), 'pos_rate', y.mean())

**Why this works.** # Classical NLP Methods — Instructor Lab

Solutions + short rationales. The goal is correctness, determinism, and interview-ready explanations.

## Dataset (offline)
Rationale: keep the focus on methods; real datasets add plumbing that hides the core signal.

## Tokenization (with n-grams)
Rationale: for classical methods, simple deterministic tokenization is often enough; n-grams can recover some word order.

## Task 2: Implement `tokenize(text, ngram=(1,1))` that:
*Section: Preprocessing*

## Section 0 — Preprocessing

### Task 0.1
Implement `tokenize(text, ngram=(1,1))` that:
- lowercases
- extracts alphabetic tokens
- optionally produces n-grams (unigrams/bigrams)

Hint: use regex `[a-z]+` and build bigrams as `w_i + '_' + w_{i+1}`.

**Hints**
- use regex `[a-z]+` and build bigrams as `w_i + '_' + w_{i+1}`.

In [ ]:
_WORD_RE = re.compile(r'[a-z]+')

def tokenize(text: str, ngram=(1, 1)) -> list[str]:
    text = text.lower()
    words = _WORD_RE.findall(text)
    n_min, n_max = ngram
    out = []
    for n in range(n_min, n_max + 1):
        if n == 1:
            out.extend(words)
        else:
            for i in range(len(words) - n + 1):
                out.append('_'.join(words[i:i+n]))
    return out

check('tokenize_basic', tokenize('Hello, WORLD!!') == ['hello', 'world'])
check('tokenize_bigrams', tokenize('a b c', ngram=(1,2)) == ['a','b','c','a_b','b_c'])

In [ ]:
# Checks
check('tokenize_basic', tokenize('Hello, WORLD!!') == ['hello', 'world'])
check('tokenize_bigrams', tokenize('a b c', ngram=(1,2)) == ['a','b','c','a_b','b_c'])

**Why this works.** ## Tokenization (with n-grams)
Rationale: for classical methods, simple deterministic tokenization is often enough; n-grams can recover some word order.

## Task 3: Create a deterministic train/val split and define evaluation helpers.
*Section: Split + metrics*

## Section 1 — Split + metrics

### Task 1.1
Create a deterministic train/val split.

Also define:
- `accuracy(y_true, y_pred)`
- `confusion_2x2(y_true, y_pred)` for binary classification

Hint: permute indices with `rng.permutation(n)`.

**Hints**
- permute indices with `rng.permutation(n)`.

In [ ]:
n = len(texts)
perm = rng.permutation(n)
n_train = int(0.75 * n)
train_idx = perm[:n_train]
val_idx = perm[n_train:]

X_train_text = [texts[i] for i in train_idx]
y_train = y[train_idx]
X_val_text = [texts[i] for i in val_idx]
y_val = y[val_idx]

def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float((y_true == y_pred).mean())

def confusion_2x2(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    cm = np.zeros((2,2), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm

check('split_disjoint', set(train_idx).isdisjoint(set(val_idx)))
print('train', len(X_train_text), 'val', len(X_val_text))

In [ ]:
# Checks
check('split_disjoint', set(train_idx).isdisjoint(set(val_idx)))

## Task 4: Compute a majority-class baseline.
*Section: Baselines*

## Section 2 — Baselines

### Task 2.1
Compute a baseline that predicts the majority class in train.

This gives a sanity-check floor before comparing richer text models.

In [ ]:
maj = int(np.round(y_train.mean()))
yhat = np.full_like(y_val, maj)
print('majority', maj, 'val_acc', accuracy(y_val, yhat))
print(confusion_2x2(y_val, yhat))

**Why this works.** ## Majority baseline
Rationale: always compare against the trivial class-frequency baseline before trusting any learned model.

## Task 5: Implement `build_vocab(...)` and `vectorize_counts(...)`.
*Section: Bag-of-Words (counts)*

## Section 3 — Bag-of-Words (counts)

### Task 3.1
Implement:
- `build_vocab(texts, min_df, ngram)` using document frequency
- `vectorize_counts(texts, vocab, ngram)` returning a dense count matrix

Hint: compute DF from a set of tokens per document.

In [ ]:
def build_vocab(texts: list[str], *, min_df: int = 1, ngram=(1,1)) -> dict[str, int]:
    df = Counter()
    for s in texts:
        df.update(set(tokenize(s, ngram=ngram)))
    kept = sorted([t for t, c in df.items() if c >= min_df])
    return {t: i for i, t in enumerate(kept)}

def vectorize_counts(texts: list[str], vocab: dict[str, int], *, ngram=(1,1)) -> np.ndarray:
    X = np.zeros((len(texts), len(vocab)), dtype=np.float64)
    for i, s in enumerate(texts):
        for tok in tokenize(s, ngram=ngram):
            j = vocab.get(tok)
            if j is not None:
                X[i, j] += 1.0
    return X

vocab = build_vocab(X_train_text, min_df=1, ngram=(1,1))
Xtr = vectorize_counts(X_train_text, vocab)
Xva = vectorize_counts(X_val_text, vocab)
print('V', len(vocab), 'Xtr', Xtr.shape)

**Why this works.** ## BoW vectorizer
Rationale: build vocabulary from train only; compute counts efficiently.

## Task 6: Implement `fit_tfidf(...)`, query vectorization, and cosine retrieval helpers.
*Section: TF-IDF + retrieval*

## Section 4 — TF-IDF + retrieval

### Task 4.1
Implement:
- `fit_tfidf(X_counts)`
- `tfidf_vectorize_query(query, vocab, idf, ngram)`
- `top_k_similar(query, X_tfidf, texts, vocab, idf, k, ngram)`

Use L2 normalization so cosine similarity reduces to a dot product.

Why L2 normalization matters:

Suppose the vocabulary is `great, acting, story`.

- Document A: `great acting story` -> TF-IDF `[1, 1, 1]`
- Document B: `great acting story great acting story great acting story` -> TF-IDF `[3, 3, 3]`

Without normalization, Document B looks artificially larger just because it repeats the same direction more times. L2 normalization removes that length effect so cosine similarity focuses on direction rather than magnitude.

**Hints**
- compute a set of tokens per document for DF.

In [ ]:
$f

In [ ]:
# Checks
check('row_norm_train', float(np.max(np.abs(np.linalg.norm(Xtr_tfidf, axis=1) - 1.0))) < 1e-6)

**Why this works.** ## TF-IDF + retrieval
Rationale: IDF downweights ubiquitous tokens; L2 normalization makes cosine similarity = dot product.

## Task 7: Implement `fit_nb(...)` and `predict_nb(...)`.
*Section: Multinomial Naive Bayes (from scratch)*

## Section 5 — Multinomial Naive Bayes (from scratch)

### Task 5.1
Implement `fit_nb(X_counts, y, alpha)` with Laplace smoothing and `predict_nb(...)`.

Hints:
- model token counts per class
- work in log-space for numerical stability

In [ ]:
def fit_nb(X_counts: np.ndarray, y: np.ndarray, alpha: float = 1.0):
    n, V = X_counts.shape
    log_prior = np.zeros((2,), dtype=np.float64)
    log_likelihood = np.zeros((2, V), dtype=np.float64)
    for c in (0, 1):
        mask = (y == c)
        Xc = X_counts[mask]
        log_prior[c] = math.log((mask.sum() + 1.0) / (n + 2.0))
        wc = Xc.sum(axis=0)
        log_likelihood[c] = np.log((wc + alpha) / (wc.sum() + alpha * V))
    return log_prior, log_likelihood

def predict_nb(X_counts: np.ndarray, log_prior: np.ndarray, log_likelihood: np.ndarray) -> np.ndarray:
    s0 = log_prior[0] + X_counts @ log_likelihood[0]
    s1 = log_prior[1] + X_counts @ log_likelihood[1]
    return (s1 > s0).astype(np.int64)

lp, ll = fit_nb(Xtr, y_train, alpha=1.0)
yhat_nb = predict_nb(Xva, lp, ll)
print('NB val_acc', accuracy(y_val, yhat_nb))
print(confusion_2x2(y_val, yhat_nb))

**Why this works.** ## Multinomial Naive Bayes
Rationale: conditional independence assumption is wrong but often yields strong decision boundaries with BoW.

## Task 8: Train a logistic regression classifier on TF-IDF features.
*Section: Logistic Regression (simple batch GD)*

## Section 6 — Logistic Regression (simple batch GD)

### Task 6.1
Train a logistic regression classifier on TF-IDF features.

Hints:
- sigmoid: `1/(1+exp(-z))`
- loss: binary cross-entropy
- gradient: `X.T @ (p - y) / n`

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

def fit_logreg(X: np.ndarray, y: np.ndarray, lr: float = 0.5, steps: int = 600, l2: float = 1e-3):
    n, d = X.shape
    w = np.zeros((d,), dtype=np.float64)
    b = 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)
        w -= lr * ((X.T @ (p - y)) / n + l2 * w)
        b -= lr * float((p - y).mean())
    return w, b

def predict_logreg(X: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    return (sigmoid(X @ w + b) >= 0.5).astype(np.int64)

w, b = fit_logreg(Xtr_tfidf, y_train)
yhat_lr = predict_logreg(Xva_tfidf, w, b)
print('LR val_acc', accuracy(y_val, yhat_lr))
print(confusion_2x2(y_val, yhat_lr))

**Why this works.** ## Logistic Regression on TF-IDF
Rationale: TF-IDF + linear classifier is a strong classical NLP baseline; L2 regularization stabilizes high-dimensional training.

## Task 9: Compare unigram vs unigram+bigram features with NB and LR.
*Section: N-grams*

## Section 7 — N-grams

### Task 7.1
Rebuild features using `ngram=(1,2)` and compare Naive Bayes and Logistic Regression against unigram-only features.

Track how vocabulary size changes and whether short phrase information helps.

**Hints**
- `df` is the number of docs where count>0.

In [ ]:
vocab12 = build_vocab(X_train_text, min_df=1, ngram=(1,2))
Xtr12 = vectorize_counts(X_train_text, vocab12, ngram=(1,2))
Xva12 = vectorize_counts(X_val_text, vocab12, ngram=(1,2))

lp12, ll12 = fit_nb(Xtr12, y_train, alpha=1.0)
yhat_nb12 = predict_nb(Xva12, lp12, ll12)

idf12, Xtr12_tfidf = fit_tfidf(Xtr12)
Xva12_tfidf = Xva12 * idf12
Xva12_tfidf = Xva12_tfidf / np.maximum(np.linalg.norm(Xva12_tfidf, axis=1, keepdims=True), 1e-12)
w12, b12 = fit_logreg(Xtr12_tfidf, y_train)
yhat_lr12 = predict_logreg(Xva12_tfidf, w12, b12)

print('V unigram', len(vocab), 'V (1,2)', len(vocab12))
print('NB (1,2) val_acc', accuracy(y_val, yhat_nb12))
print('LR (1,2) val_acc', accuracy(y_val, yhat_lr12))

**Why this works.** ## N-grams (comparison)
Rationale: bigrams add local order information, often improving phrase-sensitive tasks at the cost of a larger vocabulary.